# Gemma 4 on Amazon Bedrock Mantle — end to end

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Google DeepMind's Gemma 4 family (Apache 2.0) on the `bedrock-mantle` endpoint:
simple inference → streaming → reasoning → stateful chat → tool use →
structured JSON → multimodal → production hardening.

**Three variants, interleaved throughout this notebook:**

| Model ID | Architecture | Params | Context |
|---|---|---|---|
| `google.gemma-4-31b` | Dense | 30.7B | 256K |
| `google.gemma-4-26b-a4b` | Mixture-of-Experts | 25.2B total / 3.8B active | 256K |
| `google.gemma-4-e2b` | Dense (Per-Layer Embeddings) | 5.1B total / 2.3B effective | 128K |

**Gemma 4 is `bedrock-mantle`-only.** There is no `bedrock-runtime` support —
`invoke_model` and `converse` return errors for these model IDs.

## Self-contained, but see also
This notebook stands alone. For deeper background:
- **Auth, the three URL paths, model discovery** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Projects, cost attribution, data retention/ZDR, CloudWatch** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas, retries, service tiers, TTFT (time-to-first-token) benchmarking** →
  `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

You also need AWS credentials with `bedrock-mantle:CreateInference` and
`bedrock-mantle:CallWithBearerToken` (the managed policy
`AmazonBedrockMantleInferenceAccess` grants both).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `function_calls` | function-call items from a Responses payload |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `redact_ids` | shortens opaque service IDs (`resp_…`, `proj_…`) before they reach committed output |
| `response_text` | assistant text from a Responses API payload |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |
| `stream_lines` | raw SSE lines from a streaming endpoint, no SDK |
| `endpoints_for` | answers "mantle, runtime, or both" for a model, from the live catalogues |
| `runtime_models` | the serverless `bedrock-runtime` catalogue with modalities and inference types |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import base64
import json
import sys
import time

sys.path.insert(0, "../_shared")
from bedrock import (
    SLIDE_CALLOUTS,
    SLIDE_TITLE,
    err,
    function_calls,
    keyword_recall,
    parse_json_lenient,
    post,
    redact_ids,
    response_text,
    safe_print,
    slide_data_url,
    stream_lines,
)

# Gemma 4 is available in ALL FOUR mantle Regions — the only family that is.
REGION = "us-east-1"

DENSE = "google.gemma-4-31b"
MOE = "google.gemma-4-26b-a4b"
COMPACT = "google.gemma-4-e2b"

# NOTE the "/openai" prefix. Gemma 4's model card calls this out explicitly:
# its paths differ from the bare /v1 used by most other mantle models.
PREFIX = "/openai/v1"
BASE_URL = f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}"
print("base URL:", BASE_URL)

base URL: https://bedrock-mantle.us-east-1.api.aws/openai/v1


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "google.gemma-4-26b-a4b",
    "google.gemma-4-31b",
    "google.gemma-4-e2b",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


google.gemma-4-26b-a4b                 -- not on runtime --                     mantle


google.gemma-4-31b                     -- not on runtime --                     mantle


google.gemma-4-e2b                     -- not on runtime --                     mantle



=> 0/3 of these are on bedrock-runtime; 0 under a different id.
   bedrock-mantle only: ['google.gemma-4-26b-a4b', 'google.gemma-4-31b', 'google.gemma-4-e2b']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. Simple inference with the Responses API

The Responses API is AWS's recommended surface for new applications, and for
Gemma 4 specifically it is the **only** way to read reasoning output (see §3).

Auth here uses a short-term Bedrock API key minted from ambient IAM credentials.
(Full explanation, including the SigV4 (AWS Signature Version 4) alternative that
needs no key at all, is
in `../00-foundations/01`.)

In [3]:
from aws_bedrock_token_generator import provide_token
from openai import OpenAI

# Build the client from a FRESH token. Tokens last <=12h and cannot be refreshed,
# so don't construct one at import time and reuse it for hours.
client = OpenAI(api_key=provide_token(region=REGION), base_url=BASE_URL)

response = client.responses.create(
    model=DENSE,
    input="Explain what a mixture-of-experts model is, in two sentences.",
    max_output_tokens=200,
)
print(response.output_text)

A mixture-of-experts (MoE) model is a neural network architecture that divides its parameters into multiple specialized "expert" networks. A gating mechanism decides which specific experts to activate for a given input, allowing the model to increase its total capacity while only using a fraction of its parameters for each calculation.


## 2. Sampling parameters — probe them, every time

Gemma 4's sampling surface is the clearest example in this collection of why these
notebooks probe instead of assert. It has changed twice:

- Until **12 August 2026** it accepted `max_tokens`, the full `temperature` range
  and `top_p`.
- On 12 August it tightened to current OpenAI semantics — `max_tokens` refused in
  favour of `max_completion_tokens`, `temperature` pinned to its default, `top_p`
  rejected. That broke working code overnight, with no release note.
- It has since been **relaxed again**.

So the useful thing here is not a table of accepted values — any such table is a
snapshot that may already be wrong by the time you read it. The useful thing is the
sweep below, which asks the endpoint and derives its own conclusion.

Two facts about this model *are* durable and worth keeping in mind whichever way
the surface is set today:

- `max_output_tokens` has a **minimum of 16** on the Responses API (§2b).
- Greedy decoding at `temperature=0` sends Gemma 4 into repetition loops involving
  reserved vocabulary tokens, so even when `0.0` is accepted it is a poor choice.
  Google's and AWS's guidance is to leave `temperature` at its default of `1.0`.

Sweep the values and read what comes back:

In [4]:
print(f"{'temperature':>12} {'HTTP':>6}  detail")
print("-" * 74)
accepted, refused = [], []
for value in (0.0, 0.2, 0.5, 0.7, 0.9, 1.0, 1.5):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "Reply with exactly: OK",
            "max_output_tokens": 16,
            "temperature": value,
        },
        region=REGION,
    )
    (accepted if code == 200 else refused).append(value)
    print(f"{value:>12} {code:>6}  {'' if code == 200 else err(data)[:52]}")

code, data = post(
    f"{PREFIX}/responses",
    {"model": DENSE, "input": "Reply OK", "max_output_tokens": 16, "top_p": 0.95},
    region=REGION,
)
top_p_ok = code == 200
print(f"\n  top_p=0.95        -> HTTP {code} {'' if top_p_ok else err(data)[:70]}")

# The verdict is DERIVED, not written down. A hardcoded conclusion here is how the
# earlier version of this notebook ended up asserting 400s above a table of 200s.
print("\n--- what this run actually shows ---")
if len(accepted) == 1:
    print(f"  temperature: only {accepted[0]} accepted -> you cannot tune it on this API")
elif refused:
    print(f"  temperature: accepted {accepted}, refused {refused}")
else:
    print(f"  temperature: every value tried was accepted {accepted}")
print(f"  top_p      : {'accepted' if top_p_ok else 'rejected'}")
if not refused and top_p_ok:
    print("  => sampling is fully tunable on Responses for this model today.")
else:
    print("  => sampling is restricted today; omit what is refused rather than")
    print("     hardcoding a value, because this surface has moved before.")

 temperature   HTTP  detail
--------------------------------------------------------------------------


         0.0    200  


         0.2    200  


         0.5    200  


         0.7    200  


         0.9    200  


         1.0    200  


         1.5    200  



  top_p=0.95        -> HTTP 200 

--- what this run actually shows ---
  temperature: every value tried was accepted [0.0, 0.2, 0.5, 0.7, 0.9, 1.0, 1.5]
  top_p      : accepted
  => sampling is fully tunable on Responses for this model today.


Read the derived verdict, not the paragraph above it — that is the whole point of
keeping the probe in the notebook rather than only its result.

Whatever the surface allows today, **omitting both parameters** is the portable
choice: it works on every mantle model, and it cannot break when the accepted set
changes.

In [5]:
# The parameter surface on Chat Completions is NOT necessarily the same as on
# Responses, and it is not stable over time: Gemma 4 tightened on 12 Aug 2026 and
# has since been relaxed. So probe it rather than trusting a table -- including
# the tables in this notebook.
probes = [
    ("max_tokens=16", {"max_tokens": 16}),
    ("max_completion_tokens=16", {"max_completion_tokens": 16}),
    ("+ temperature=0.2", {"max_completion_tokens": 16, "temperature": 0.2}),
    ("+ temperature=1.0", {"max_completion_tokens": 16, "temperature": 1.0}),
    ("+ top_p=0.95", {"max_completion_tokens": 16, "top_p": 0.95}),
]
print(f"{'parameters':<26} {'HTTP':<5} detail")
print("-" * 76)
cc = {}
for label, extra in probes:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": DENSE,
            "messages": [{"role": "user", "content": "Reply OK"}],
            **extra,
        },
        region=REGION,
    )
    cc[label] = code
    print(f"{label:<26} {code:<5} {'' if code == 200 else err(data)[:42]}")

# Derive the summary from `cc` so it can never contradict the table above.
print()
budget = [k for k in ("max_tokens=16", "max_completion_tokens=16") if cc[k] == 200]
print(f"=> token budget accepted as: {', '.join(b.split('=')[0] for b in budget)}")
sampling_ok = [k for k in probes[2:] if cc[k[0]] == 200]
if len(sampling_ok) == 3:
    print("   temperature and top_p are both accepted here.")
else:
    refused = [k[0].lstrip('+ ') for k in probes[2:] if cc[k[0]] != 200]
    print(f"   refused: {', '.join(refused)}")
print("   Compare with section 2: the two APIs do not have to agree, and neither")
print("   set is stable, so gate sampling per model AND re-probe periodically.")

parameters                 HTTP  detail
----------------------------------------------------------------------------


max_tokens=16              200   


max_completion_tokens=16   200   


+ temperature=0.2          200   


+ temperature=1.0          200   


+ top_p=0.95               200   

=> token budget accepted as: max_tokens, max_completion_tokens
   temperature and top_p are both accepted here.
   Compare with section 2: the two APIs do not have to agree, and neither
   set is stable, so gate sampling per model AND re-probe periodically.


In [6]:
# Another undocumented constraint: max_output_tokens has a MINIMUM of 16.
for n in (8, 15, 16):
    code, data = post(
        f"{PREFIX}/responses",
        {"model": DENSE, "input": "Hi", "max_output_tokens": n},
        region=REGION,
    )
    detail = "" if code == 200 else err(data)[:70]
    print(f"  max_output_tokens={n:3} -> HTTP {code} {detail}")

  max_output_tokens=  8 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


  max_output_tokens= 15 -> HTTP 400 Invalid 'max_output_tokens': integer below minimum value. Expected a v


  max_output_tokens= 16 -> HTTP 200 


## 3. Reasoning — and why the API choice matters

All three variants have built-in reasoning. The critical detail from the model
card: reasoning effort is honoured on **both** Responses and Chat Completions,
and the model does the extended thinking either way — but **only the Responses
API returns the reasoning content**. On Chat Completions you pay for those
tokens and never see them.

In [7]:
resp = client.responses.create(
    model=DENSE,
    input=(
        "A train leaves at 3pm travelling 60 km/h. Another leaves an hour later "
        "at 90 km/h from the same station on the same track. When does the second "
        "catch the first?"
    ),
    reasoning={"effort": "high"},
    max_output_tokens=1200,
)

reasoning_blocks = []
for item in resp.output:
    if item.type == "reasoning":
        for block in item.content:
            text = getattr(block, "text", "")
            if text:
                reasoning_blocks.append(text)

print("=== REASONING (visible only on the Responses API) ===")
print(("\n".join(reasoning_blocks))[:700] or "(none returned)")
print("\n=== FINAL ANSWER ===")
print(resp.output_text[:400])
print("\nreasoning tokens:", resp.usage.output_tokens_details.reasoning_tokens)

=== REASONING (visible only on the Responses API) ===
*   Train A: Departs at 3:00 PM, speed = 60 km/h.
    *   Train B: Departs at 4:00 PM (one hour later), speed = 90 km/h.
    *   Both trains leave from the same station on the same track.
    *   Goal: Find the time when Train B catches Train A.

    *   Let $t$ be the time in hours that Train B has been traveling.
    *   Since Train A left one hour earlier, Train A has been traveling for $(t + 1)$ hours.

    *   Distance = Speed $\times$ Time.
    *   Distance of Train A: $D_A = 60(t + 1)$
    *   Distance of Train B: $D_B = 90(t)$

    *   Train B catches Train A when their distances from the station are equal.
    *   $60(t + 1) = 90t$

    *   $60t + 60 = 90t$
    *   Subtract $60t$ fr

=== FINAL ANSWER ===
The second train will catch the first at **6:00 PM**.

Here is the step-by-step breakdown:

**1. Determine the lead distance**
The first train leaves at 3:00 PM and travels at 60 km/h. Since the second train doesn't leave u

In [8]:
# Which effort values are valid? Probe rather than assume.
for effort in ("none", "minimal", "low", "medium", "high"):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "2+2?",
            "max_output_tokens": 32,
            "reasoning": {"effort": effort},
        },
        region=REGION,
    )
    print(f"  effort={effort:8} -> HTTP {code} {'' if code == 200 else err(data)[:60]}")

  effort=none     -> HTTP 200 


  effort=minimal  -> HTTP 400 Unsupported value: 'minimal' is not supported with the 'goog


  effort=low      -> HTTP 200 


  effort=medium   -> HTTP 200 


  effort=high     -> HTTP 200 


`minimal` is rejected; the valid ladder is `none` / `low` / `medium` / `high`.

**Variant-specific advice:** for `gemma-4-e2b`, set `effort="high"`. The smallest
variant reasons extensively by default, and high effort keeps that thinking in
the dedicated reasoning channel instead of leaking into the final answer.

In [9]:
# Measure the TRACE, not the token counter. Gemma 4 reports
# output_tokens_details.reasoning_tokens as 0 even when it returns a full
# reasoning item, so counting tokens here shows nothing and reads as "it did not
# think". Count the characters the model actually returned.
for model in (COMPACT, DENSE):
    r = client.responses.create(
        model=model,
        input="If 3 shirts dry in 4 hours, how long for 9 shirts on the same line?",
        reasoning={"effort": "high"},
        max_output_tokens=1200,
    )
    trace = "".join(
        getattr(block, "text", "") or ""
        for item in r.output
        if item.type == "reasoning"
        for block in (item.content or [])
    )
    print(
        f"{model:26} trace={len(trace):5} chars  "
        f"reasoning_tokens={r.usage.output_tokens_details.reasoning_tokens} "
        f"(never itemised)"
    )
    print(f"{'':26} answer={r.output_text[:80]!r}")

google.gemma-4-e2b         trace= 1570 chars  reasoning_tokens=0 (never itemised)
                           answer='This is a direct proportion problem. If you have more shirts, it will take more '


google.gemma-4-31b         trace=  770 chars  reasoning_tokens=0 (never itemised)
                           answer='**4 hours.**\n\nSince they are all on the same line, they dry at the same time.'


## 4. Streaming

Reasoning and answer text arrive on **separate event types** — that's what lets
you render a "thinking…" panel distinct from the answer.

In [10]:
# effort="high" and a generous budget on purpose: at low effort this model emits
# no reasoning deltas at all, and 400 tokens truncates the answer mid-word -- so
# the earlier version of this cell demonstrated neither of the two event types it
# claims to separate.
stream = client.responses.create(
    model=DENSE,
    input="List three properties of a good distributed queue, one line each.",
    reasoning={"effort": "high"},
    max_output_tokens=2000,
    stream=True,
)

event_counts = {}
print("--- live stream ---")
try:
    for event in stream:
        event_counts[event.type] = event_counts.get(event.type, 0) + 1
        if event.type == "response.reasoning_text.delta":
            print("\033[2m" + event.delta + "\033[0m", end="", flush=True)
        elif event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)
except Exception as exc:
    # A stream can fail AFTER delivering part of the answer: a mid-stream
    # 5xx is not rare, and it has happened while building these notebooks.
    # Report what arrived instead of losing it - production code has to
    # decide whether a partial answer is usable or the call must be retried.
    print(f"\n[stream interrupted after the deltas above: {type(exc).__name__}]")
print("\n\n--- event types seen ---")
for name, count in sorted(event_counts.items(), key=lambda kv: -kv[1]):
    print(f"  {count:4}  {name}")

# Derived, so it cannot claim a separation the run did not show.
reasoning_deltas = event_counts.get("response.reasoning_text.delta", 0)
text_deltas = event_counts.get("response.output_text.delta", 0)
print(f"\nreasoning deltas={reasoning_deltas}  text deltas={text_deltas}")
if reasoning_deltas and text_deltas:
    print("=> two distinct event types, so a 'thinking' panel can be rendered")
    print("   separately from the answer.")
elif text_deltas:
    print("=> only text deltas this run. The reasoning channel is not guaranteed:")
    print("   it depends on effort and on the prompt, so handle its absence.")
if event_counts.get("response.incomplete"):
    print("=> response.incomplete: the budget ran out. Raise max_output_tokens.")

--- live stream ---


Scal

ability

:

 Ability

...

 individual

 node

 crashes

.



--- event types seen ---
   291  response.reasoning.delta
    42  response.output_text.delta
     2  response.output_item.added
     2  response.output_item.done
     1  response.created
     1  response.in_progress
     1  response.reasoning.done
     1  response.content_part.added
     1  response.output_text.done
     1  response.content_part.done
     1  response.completed

reasoning deltas=0  text deltas=42
=> only text deltas this run. The reasoning channel is not guaranteed:
   it depends on effort and on the prompt, so handle its absence.


## 5. Multi-turn: two approaches

### (a) Send the history yourself
`input` accepts the same role/content array that Chat Completions calls
`messages`. Fully stateless — nothing is retained server-side.

In [11]:
conversation = [
    {"role": "system", "content": "You are terse. Answer in one short sentence."},
    {"role": "user", "content": "What is a MoE model?"},
]
first = client.responses.create(model=MOE, input=conversation, max_output_tokens=120)
print("assistant:", first.output_text)

conversation += [
    {"role": "assistant", "content": first.output_text},
    {"role": "user", "content": "And why is it cheaper to run?"},
]
second = client.responses.create(model=MOE, input=conversation, max_output_tokens=120)
print("assistant:", second.output_text)

assistant: A Mixture of Experts (MoE) model uses a sparse architecture where only specific sub-networks are activated for each input.


assistant: It is cheaper because it only activates a fraction of its total parameters for each calculation, reducing computational cost.


**Important for Gemma 4:** append only the *final answers* to history, never the
reasoning items. AWS warns that replaying prior reasoning back to the model
degrades later turns. Keep reasoning in your logs, strip it from `input`.

### (b) Server-side state with `previous_response_id`
Bedrock rebuilds the context for you. Cheaper on input tokens for long chats —
but it requires `store=True`, which retains input and output for **30 days**
in-Region (encrypted, project-scoped).

In [12]:
turn1 = client.responses.create(
    model=DENSE,
    input="My favourite database is DynamoDB. Reply with just: noted.",
    max_output_tokens=32,
    store=True,
)
print("turn 1 id:", redact_ids(turn1.id))

turn2 = client.responses.create(
    model=DENSE,
    input="What is my favourite database?",
    previous_response_id=turn1.id,
    max_output_tokens=48,
)
print("turn 2   :", turn2.output_text)

turn 1 id: resp_eaa55e3x...


turn 2   : Your favourite database is DynamoDB.


In [13]:
# The privacy/convenience trade-off, made concrete.
private = client.responses.create(
    model=DENSE, input="Secret: 42. Reply: ok.", max_output_tokens=16, store=False
)
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "What was the secret?",
        "previous_response_id": private.id,
        "max_output_tokens": 32,
    },
    region=REGION,
)
print(f"chaining from a store=False response -> HTTP {code}")
print("message:", err(data)[:100])
print("\n=> Choose: server-side state (store=True) OR zero retention, not both.")

chaining from a store=False response -> HTTP 404
message: Response not found.

=> Choose: server-side state (store=True) OR zero retention, not both.


## 6. Retrieve, and run in the background

Stored responses can be fetched later, and long jobs can run detached.

In [14]:
code, fetched = post(
    f"{PREFIX}/responses/{turn1.id}", None, region=REGION, method="GET"
)
print(f"GET  -> {code} status={fetched.get('status')}")

bg = client.responses.create(
    model=DENSE,
    input="Write a short paragraph about idempotency in distributed systems.",
    max_output_tokens=300,
    background=True,
    store=True,
)
print("background job:", redact_ids(bg.id), "status:", bg.status)

for _ in range(30):
    time.sleep(2)
    code, polled = post(
        f"{PREFIX}/responses/{bg.id}", None, region=REGION, method="GET"
    )
    if polled.get("status") in ("completed", "failed", "cancelled"):
        break
print("final status:", polled.get("status"))
print("text:", response_text(polled)[:200])

GET  -> 200 status=completed


background job: resp_fmkt6cjt... status: in_progress


final status: completed
text: Idempotency in distributed systems is the property where an operation can be applied multiple times without changing the result beyond the initial application. In a distributed environment, network fa


In [15]:
# Tidy up the stored responses we created.
for rid in (turn1.id, bg.id):
    code, _ = post(f"{PREFIX}/responses/{rid}", None, region=REGION, method="DELETE")
    print(f"DELETE {redact_ids(rid)} -> {code}")

DELETE resp_eaa55e3x... -> 200


DELETE resp_fmkt6cjt... -> 200


## 7. A short detour to Chat Completions

Gemma 4 supports Chat Completions too (same `/openai/v1` prefix). Reach for it
when you have existing OpenAI-shaped code, or want the simpler stateless model.

Two differences worth seeing side by side: `messages` instead of `input`, and
`max_completion_tokens` accepted alongside `max_output_tokens`.

**The reasoning trace is not simply absent here.** At `reasoning_effort="high"`
Gemma 4 returns it in a non-standard `choices[0].message.reasoning` field — a
sibling of `content`, not part of it, and not in the OpenAI specification. At
`none` and `low` the key is absent entirely. `output_tokens_details.reasoning_tokens`
is `0` in every case, so the trace is readable but never itemised: you cannot
separate thinking from answering in the usage figures. The same non-standard field
carries Qwen's and DeepSeek's traces, so treat it as a mantle convention rather
than a Gemma quirk (`../14-openai-gpt-oss/01` covers it for gpt-oss).

> **This surface has changed twice.** Gemma 4 accepted `max_tokens`, a full
> `temperature` range and `top_p` until 12 August 2026; it then tightened to
> current OpenAI semantics with no release note; it has since been relaxed again.
> The probes in sections 2 and 2b ask the live endpoint and derive their own
> verdicts, so you see today's answer. Treat every parameter table in every sample
> — including these — as a snapshot.


In [16]:
cc = client.chat.completions.create(
    model=DENSE,
    messages=[
        {"role": "system", "content": "You are terse."},
        {"role": "user", "content": "Why is idempotency useful? One sentence."},
    ],
    max_completion_tokens=120,  # max_tokens is rejected -- see section 3
)
print("content:", cc.choices[0].message.content)
print("\nusage:", cc.usage.model_dump_json())


content: Idempotency ensures that performing the same operation multiple times produces the same result as a single execution, preventing duplicate actions and ensuring system consistency during retries.

usage: {"completion_tokens":32,"prompt_tokens":47,"total_tokens":79,"completion_tokens_details":{"accepted_prediction_tokens":0,"audio_tokens":0,"reasoning_tokens":0,"rejected_prediction_tokens":0},"prompt_tokens_details":{"audio_tokens":0,"cache_write_tokens":0,"cached_tokens":0}}


In [17]:
# Reasoning still costs tokens on Chat Completions, but the OpenAI CC schema has
# nowhere to return the trace -- you pay for it and cannot read it.
code, data = post(
    f"{PREFIX}/chat/completions",
    {
        "model": DENSE,
        "messages": [{"role": "user", "content": "Tricky: 17*23?"}],
        "max_completion_tokens": 300,
        "reasoning_effort": "high",
    },
    region=REGION,
)
message = data.get("choices", [{}])[0].get("message", {}) or {}
details = (data.get("usage", {}) or {}).get("completion_tokens_details") or {}
trace = message.get("reasoning") or ""
print(f"HTTP {code} | keys in message: {sorted(message.keys())}")
print(f"message.reasoning     : {len(trace)} chars  <- NOT in the OpenAI schema")
print("reasoning tokens billed:", details.get("reasoning_tokens"), "(never itemised)")
print("answer:", (message.get("content") or "")[:80])
if trace:
    print("\n=> The trace IS returned here, in a non-standard field. Code written")
    print("   against the published OpenAI schema reads .content only and discards")
    print("   it. Read .reasoning explicitly if you want it.")
else:
    print("\n=> No trace at this effort level. Try reasoning_effort='high'.")

# Whether function tools coexist with reasoning is a per-model, per-date fact:
# Gemma 4 refused the combination in August 2026 and no longer does. Probe both.
TOOL = [
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two integers",
            "parameters": {
                "type": "object",
                "properties": {"a": {"type": "integer"}, "b": {"type": "integer"}},
                "required": ["a", "b"],
            },
        },
    }
]
print()
tool_results = {}
for label, extra in [
    ("tools alone", {}),
    ('tools + reasoning_effort "none"', {"reasoning_effort": "none"}),
    ('tools + reasoning_effort "high"', {"reasoning_effort": "high"}),
]:
    code, data = post(
        f"{PREFIX}/chat/completions",
        {
            "model": DENSE,
            "messages": [{"role": "user", "content": "What is 17*23? Use the tool."}],
            "max_completion_tokens": 200,
            "tools": TOOL,
            **extra,
        },
        region=REGION,
    )
    choice = data.get("choices", [{}])[0].get("message", {}) or {}
    calls = choice.get("tool_calls") or []
    detail = "" if code == 200 else err(data)[:46]
    tool_results[label] = (code, len(calls))
    print(f"  {label:<36} HTTP {code} tool_calls={len(calls)} {detail}")

# Derived verdict. Hardcoding this sentence is exactly how the previous version of
# the notebook came to assert a 400 that the service had stopped returning.
print()
refused = [k for k, (c, _) in tool_results.items() if c != 200]
if not refused:
    print("=> Tools work at every reasoning effort on this model today.")
elif refused == ["tools alone"]:
    print('=> Tools require reasoning_effort to be set explicitly on this model.')
else:
    print(f"=> Refused combinations today: {refused}")
    print('   The message names the fix when it is a reasoning conflict.')


HTTP 200 | keys in message: ['annotations', 'content', 'reasoning', 'refusal', 'role']
message.reasoning     : 475 chars  <- NOT in the OpenAI schema
reasoning tokens billed: 0 (never itemised)
answer: 17 * 23 = **391**

=> The trace IS returned here, in a non-standard field. Code written
   against the published OpenAI schema reads .content only and discards
   it. Read .reasoning explicitly if you want it.



  tools alone                          HTTP 200 tool_calls=1 


  tools + reasoning_effort "none"      HTTP 200 tool_calls=1 


  tools + reasoning_effort "high"      HTTP 200 tool_calls=1 

=> Tools work at every reasoning effort on this model today.


## 8. Tool use (function calling)

Gemma 4 has native function calling. Note the **flat** Responses tool shape —
`name`/`description`/`parameters` at the top level, unlike Chat Completions which
nests them under `"function"`.

In [18]:
def get_weather(location: str, unit: str = "celsius") -> dict:
    """Stand-in for a real weather API."""
    table = {"seattle": 12, "singapore": 31, "berlin": 8}
    celsius = table.get(location.split(",")[0].strip().lower(), 20)
    value = celsius if unit == "celsius" else round(celsius * 9 / 5 + 32)
    return {"location": location, "temperature": value, "unit": unit, "sky": "cloudy"}


weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string", "description": "City, e.g. Seattle"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
        },
        "required": ["location"],
    },
}

conv = [{"role": "user", "content": "What's the weather in Seattle?"}]
first = client.responses.create(
    model=DENSE,
    input=conv,
    tools=[weather_tool],
    tool_choice="auto",
    max_output_tokens=300,
)

calls = [i for i in first.output if i.type == "function_call"]
print("tool calls requested:", [(c.name, c.arguments) for c in calls])

for call in calls:
    args = json.loads(call.arguments)
    result = get_weather(**args)
    # Echo the call, then its result. Note: no reasoning items replayed.
    conv.append(
        {
            "type": "function_call",
            "call_id": call.call_id,
            "name": call.name,
            "arguments": call.arguments,
        }
    )
    conv.append(
        {
            "type": "function_call_output",
            "call_id": call.call_id,
            "output": json.dumps(result),
        }
    )

final = client.responses.create(
    model=DENSE, input=conv, tools=[weather_tool], max_output_tokens=200
)
print("\nfinal answer:", final.output_text)

tool calls requested: [('get_weather', '{"location":"Seattle"}')]



final answer: The weather in Seattle is currently 12°C and cloudy.


### One tool call per turn

Gemma 4's model card states parallel tool calls are not supported. It does not
error — it just quietly does one. Design your loop to iterate rather than
expecting a batch.

In [19]:
tool_a = {
    "type": "function",
    "name": "tool_a",
    "description": "Records a value for A.",
    "parameters": {
        "type": "object",
        "properties": {"x": {"type": "string"}},
        "required": ["x"],
    },
}
tool_b = {
    "type": "function",
    "name": "tool_b",
    "description": "Records a value for B.",
    "parameters": {
        "type": "object",
        "properties": {"y": {"type": "string"}},
        "required": ["y"],
    },
}

multi = client.responses.create(
    model=DENSE,
    input="Call tool_a with x='1' AND tool_b with y='2'. Both, right now.",
    tools=[tool_a, tool_b],
    max_output_tokens=300,
)
issued = [i.name for i in multi.output if i.type == "function_call"]
print(f"asked for 2 tool calls, model issued {len(issued)}: {issued}")
print("=> iterate; do not assume a batch.")

asked for 2 tool calls, model issued 1: ['tool_a']
=> iterate; do not assume a batch.


## 9. Structured JSON output

Two routes, and an important caveat about the first one.

In [20]:
# (a) Native strict schema via text.format
schema = {
    "type": "object",
    "properties": {
        "language": {"type": "string"},
        "typed": {"type": "boolean"},
        "year_created": {"type": "integer"},
    },
    "required": ["language", "typed", "year_created"],
    "additionalProperties": False,
}

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Describe the Rust programming language.",
        "max_output_tokens": 200,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "lang",
                "schema": schema,
                "strict": True,
            }
        },
    },
    region=REGION,
)
raw = response_text(data)
print("native json_schema ->", code)
print("raw output:", repr(raw))

native json_schema -> 200
raw output: '{"language":"Rust","typed":true,"year_created":2010}'


### ⚠️ "Strict" is not reliably strict on Gemma 4

Gemma 4 intermittently appends characters **after** a well-formed JSON object —
a stray `\n}`, or occasionally unrelated text. The object itself is correct, but
`json.loads()` on the whole string raises. In repeated testing this happened in
roughly half of runs.

Never call bare `json.loads()` on Gemma 4 structured output in production.

In [21]:
runs, invalid = 6, 0
for i in range(runs):
    code, data = post(
        f"{PREFIX}/responses",
        {
            "model": DENSE,
            "input": "Describe the Rust programming language.",
            "max_output_tokens": 200,
            "text": {
                "format": {
                    "type": "json_schema",
                    "name": "lang",
                    "schema": schema,
                    "strict": True,
                }
            },
        },
        region=REGION,
    )
    text = response_text(data)
    try:
        json.loads(text)
        verdict = "parses"
    except json.JSONDecodeError:
        verdict = "FAILS json.loads"
        invalid += 1
    print(f"  run {i + 1}: {verdict:18} {text!r}")

print(f"\n{invalid}/{runs} runs would crash a naive json.loads()")

  run 1: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 2: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 3: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 4: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 5: parses             '{"language":"Rust","typed":true,"year_created":2010}'


  run 6: parses             '{"language":"Rust","typed":true,"year_created":2010}'

0/6 runs would crash a naive json.loads()


In [22]:
# The fix: extract the first balanced JSON object. bedrock.parse_json_lenient()
# does exactly this, and is safe to use on every model.

for sample in [
    '{"language":"Rust","typed":true,"year_created":2010}',
    '{"language":"Rust","typed":true,"year_created":2010}\n}',
    '{"language":"Rust","typed":true,"year_created":2010}\ntrailing text',
    '```json\n{"language":"Rust","typed":true,"year_created":2010}\n```',
]:
    print(f"  {parse_json_lenient(sample)}   <- from {sample[:52]!r}")

  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '{"language":"Rust","typed":true,"year_created":2010}'
  {'language': 'Rust', 'typed': True, 'year_created': 2010}   <- from '```json\n{"language":"Rust","typed":true,"year_create'


In [23]:
# Re-run the real call and parse it safely.
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Describe the Rust programming language.",
        "max_output_tokens": 200,
        "text": {
            "format": {
                "type": "json_schema",
                "name": "lang",
                "schema": schema,
                "strict": True,
            }
        },
    },
    region=REGION,
)
parsed = parse_json_lenient(response_text(data))
print("parsed safely:")
print(json.dumps(parsed, indent=2))
expected = {"language", "typed", "year_created"}
if set(parsed) != expected:
    raise ValueError(
        f"schema mismatch: expected {sorted(expected)}, got {sorted(parsed)}"
    )
print("schema honoured exactly:", sorted(parsed))

parsed safely:
{
  "language": "Rust",
  "typed": true,
  "year_created": 2010
}
schema honoured exactly: ['language', 'typed', 'year_created']


In [24]:
# (b) Forced tool call — the arguments ARE the output. More portable: it works on
# models that lack native structured output, and enum constraints are respected.
emit = {
    "type": "function",
    "name": "emit_profile",
    "description": "Return the analysis. Use 'unknown' if unsure.",
    "parameters": {
        "type": "object",
        "properties": {
            "summary": {"type": "string"},
            "sentiment": {
                "type": "string",
                "enum": ["positive", "neutral", "negative"],
            },
            "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        },
        "required": ["summary", "sentiment", "confidence"],
    },
}

forced = client.responses.create(
    model=DENSE,
    input="Review: 'The battery life is superb but the screen scratches easily.'",
    tools=[emit],
    tool_choice={"type": "function", "name": "emit_profile"},  # must call it
    max_output_tokens=300,
)
call = next(i for i in forced.output if i.type == "function_call")
print("forced-tool output:")
# parse_json_lenient again: tool arguments can carry the same trailing garbage.
print(json.dumps(parse_json_lenient(call.arguments), indent=2))

forced-tool output:
{
  "summary": "The user is pleased with the battery life but dissatisfied with the screen's durability.",
  "sentiment": "neutral",
  "confidence": "high"
}


### Schema keywords

You may read that Gemma 4 rejects JSON-Schema constraint keywords like
`minLength` and `pattern`. On the current `/openai/v1` Responses path they are
**accepted** — verify against your own path and model before adding a sanitiser.

In [25]:
constrained = {
    "type": "function",
    "name": "emit",
    "description": "Emit a code.",
    "parameters": {
        "type": "object",
        "properties": {"code": {"type": "string", "minLength": 3, "pattern": "^[A-Z]"}},
        "required": ["code"],
    },
}
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Emit code 'ABC'.",
        "max_output_tokens": 100,
        "tools": [constrained],
    },
    region=REGION,
)
print(
    f"schema with minLength + pattern -> HTTP {code} "
    f"{'accepted' if code == 200 else err(data)[:70]}"
)

schema with minLength + pattern -> HTTP 200 accepted


## 10. Multimodal — image input

All three variants take text + image. Constraints from the model card and
testing:

- Base64 data URLs or `s3://` URLs. **Arbitrary `https://` image URLs are not
  supported.**
- Total request body max **3.5 MB**.
- Put the image **before** the text (Google's recommended ordering).
- Very small images are rejected as an unsupported format — use realistic sizes.

In [26]:
# A slide from a public AWS talk, so this is an OCR question with a checkable
# answer. See ../_shared/bedrock.py for provenance.
data_url = slide_data_url()
print("data URL bytes:", len(data_url))

vision = client.responses.create(
    model=DENSE,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_image", "image_url": data_url},  # image FIRST
                {
                    "type": "input_text",
                    "text": "Read this slide. Give its title, then quote the three "
                            "green callout lines.",
                },
            ],
        }
    ],
    max_output_tokens=300,
)
answer = vision.output_text.strip()
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
print(f"callouts {hits}/{total}, title={SLIDE_TITLE.lower() in answer.lower()}")
print(" ", " ".join(answer.split())[:200])

data URL bytes: 43607


callouts 3/3, title=True
  **Title:** Ingestion from database **Green callout lines:** * "Pay according to job duration" * "Lower compute cost, up to 90%" * "Lower storage cost for rarely accessed data"


In [27]:
# A 1x1 pixel PNG is rejected. Worth knowing so you don't chase a phantom bug.
# A literal 1x1 PNG, so the degenerate input is obvious rather than generated.
TINY_PNG_B64 = (
    "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mP8z8"
    "AAxAAADwABbT8HRAAAAABJRU5ErkJggg=="
)
code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "max_output_tokens": 16,
        "input": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "input_image",
                        "image_url": f"data:image/png;base64,{TINY_PNG_B64}",
                    },
                    {"type": "input_text", "text": "Colour?"},
                ],
            }
        ],
    },
    region=REGION,
)
print(f"1x1 PNG -> HTTP {code}: {err(data)[:80]}")

1x1 PNG -> HTTP 400: Invalid or unsupported image format


There is also an undocumented ceiling of roughly **32 images per request**.
Payloads far below the 3.5 MB size limit can still fail with
`400 / "Engine bad request"` once you exceed it. Batch large image sets.

## 11. Choosing a variant — a like-for-like comparison

Same prompt, all three variants, measuring latency and tokens.

In [28]:
task = "In one sentence, explain why eventual consistency is a useful trade-off."

print(f"{'model':26} {'latency':>9} {'reason tok':>11} {'out tok':>8}  answer")
print("-" * 104)
for model in (COMPACT, MOE, DENSE):
    started = time.perf_counter()
    r = client.responses.create(
        model=model, input=task, reasoning={"effort": "low"}, max_output_tokens=250
    )
    elapsed = time.perf_counter() - started
    details = r.usage.output_tokens_details
    print(
        f"{model:26} {elapsed:>8.2f}s {details.reasoning_tokens:>11} "
        f"{r.usage.output_tokens:>8}  {r.output_text.strip()[:44]!r}"
    )

model                        latency  reason tok  out tok  answer
--------------------------------------------------------------------------------------------------------


google.gemma-4-e2b             2.33s           0       38  'Eventual consistency is a useful trade-off b'


google.gemma-4-26b-a4b         1.07s           0       39  'Eventual consistency is a useful trade-off b'


google.gemma-4-31b            10.96s           0       43  'Eventual consistency is a useful trade-off b'


| If your workload is… | Choose | Why |
|---|---|---|
| Reasoning- or coding-heavy | `gemma-4-31b` | Largest dense variant, 256K context |
| Cost-sensitive at high throughput | `gemma-4-26b-a4b` | MoE (mixture-of-experts): ~4B-class cost, larger knowledge capacity |
| Latency-sensitive, on-device-style | `gemma-4-e2b` | Smallest and fastest; set `effort="high"` |

All three share one API surface, so you can develop once and switch by model ID.

## 12. Production hardening

Retries, cost attribution, and privacy in one place.
(Background: `../00-foundations/03-scaling-tiers-and-latency.ipynb`.)

In [29]:
# Attribute usage to a project for cost tracking (see ../00-foundations/02).
code, project = post(
    "/v1/organization/projects",
    {
        "name": "gemma4-samples",
        "tags": {"Application": "Gemma4Demo", "Environment": "Demo"},
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id)

code, data = post(
    f"{PREFIX}/responses",
    {
        "model": DENSE,
        "input": "Reply OK",
        "max_output_tokens": 16,
        "service_tier": "flex",
        "store": False,
    },
    region=REGION,
    headers={"OpenAI-Project": project_id},
)
print(f"attributed call -> HTTP {code} resolved tier={data.get('service_tier')}")

project: 200 proj_qb7z7her...


attributed call -> HTTP 200 resolved tier=flex


In [30]:
class Gemma4Client:
    """Teaching pattern: fresh token, right params, retries, attribution.
    Not production-ready as written - review and adapt before deployment."""

    def __init__(self, model=DENSE, region=REGION, tier="default", project=None):
        self.model, self.region, self.tier, self.project = model, region, tier, project

    def ask(self, prompt, *, effort="low", max_output_tokens=512, structured=None):
        body = {
            "model": self.model,
            "input": prompt,
            "max_output_tokens": max(16, max_output_tokens),  # API minimum is 16
            # temperature and top_p are deliberately omitted rather than set:
            # which values this model accepts has changed twice (see section 2),
            # and omitting them is the one choice that cannot break.
            "reasoning": {"effort": effort},
            "service_tier": self.tier,
            "store": False,  # no 30-day retention
        }
        if structured:
            body["text"] = {
                "format": {
                    "type": "json_schema",
                    "name": "out",
                    "schema": structured,
                    "strict": True,
                }
            }
        headers = {"OpenAI-Project": self.project} if self.project else None
        # post() retries 429/5xx with exponential backoff — mantle has no RPM
        # quota and sheds load under regional pressure.
        code, data = post(
            f"{PREFIX}/responses", body, region=self.region, headers=headers
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        return data


gemma = Gemma4Client(model=MOE, tier="flex", project=project_id)
out = gemma.ask(
    "Name the capital of Japan.",
    structured={
        "type": "object",
        "properties": {"capital": {"type": "string"}},
        "required": ["capital"],
        "additionalProperties": False,
    },
)
print("structured:", parse_json_lenient(response_text(out)))
print("usage:", json.dumps(out.get("usage", {})))

structured: {'capital': 'Tokyo'}
usage: {"input_tokens": 72, "input_tokens_details": {"cache_write_tokens": 0, "cached_tokens": 0}, "output_tokens": 6, "output_tokens_details": {"reasoning_tokens": 0}, "total_tokens": 78}


In [31]:
# Clean up the demo project.
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("archived project:", code, archived.get("status"))

archived project: 200 archived


## Gotchas — Gemma 4 on bedrock-mantle

| Gotcha | Detail |
|---|---|
| `bedrock-mantle` only | No `bedrock-runtime` support for these model IDs |
| Path prefix | `/openai/v1`, **not** the bare `/v1` most mantle models use |
| Sampling surface **moves** | Tightened 12 Aug 2026, relaxed since. §2 probes it and derives the verdict; omitting `temperature`/`top_p` is the choice that cannot break |
| `temperature=0` | Even where accepted, it drives repetition loops over reserved tokens — leave the default |
| `max_output_tokens` | Minimum **16**; smaller values 400 |
| `reasoning.effort` | `none`/`low`/`medium`/`high`; **`minimal` is rejected** |
| Reasoning visibility | Responses returns a `reasoning` item; Chat Completions returns `message.reasoning` at `effort="high"` only — **not** in the OpenAI schema |
| `reasoning_tokens` | Always `0` for this model, on both APIs. The trace is readable but never itemised |
| Reasoning replay | Never append reasoning items to history — degrades quality |
| Parallel tool calls | Unsupported; model silently issues one |
| Tools + reasoning | The combination was refused in Aug 2026 and is not now — §7 probes it |
| `store=False` | Blocks `previous_response_id` chaining (404) |
| Images | base64 or `s3://` only; ≤3.5 MB body; ~32 images max; 1×1 PNG rejected |
| e2b reasoning | Set `effort="high"` to keep thinking in the reasoning item rather than the answer |
| Regions | The only family in all four mantle Regions |

## Where next
- Same-API neighbours: `../01-openai-gpt/` (web search, caching), `../11-xai-grok/`
- Different API shape: `../04-qwen/` (Chat Completions), `../02-anthropic-claude/`
  (Messages)
- Cross-cutting: `../99-cross-cutting/`

## Also on `bedrock-runtime`? Gemma 4 — no

Gemma 4 is **`bedrock-mantle` only**. There is no Converse path for it today, so a workload on Gemma 4 cannot be moved to `bedrock-runtime` without changing model. Gemma **3** is on both endpoints - see the sibling notebook.

The cell below confirms it against the live catalogues rather than asserting it,
because model availability moves.


In [32]:
from bedrock import endpoints_for, runtime_models

MODEL = "google.gemma-4-31b"
where = endpoints_for(MODEL)
print(f"{MODEL} -> {where}")

if not where["runtime"]:
    print("\nNot on bedrock-runtime, so Converse is not an option for this model.")
    print("Same-provider models that ARE on bedrock-runtime today:")
    provider = MODEL.split(".")[0]
    siblings = sorted(m for m in runtime_models() if m.startswith(provider + "."))
    for sibling in siblings[:8]:
        print("   ", sibling)
    if not siblings:
        print("    (none)")


google.gemma-4-31b -> {'mantle': True, 'runtime': False}

Not on bedrock-runtime, so Converse is not an option for this model.
Same-provider models that ARE on bedrock-runtime today:
    google.gemma-3-12b-it
    google.gemma-3-27b-it
    google.gemma-3-4b-it
